# Huấn luyện mô hình U-Net (Semantic Segmentation)
Đây là thuật toán phân vùng điểm ảnh để "vẽ" tỷ lệ diện tích vết bẩn, nước đọng. Thích hợp cho data chuẩn Mendeley/Github.

In [ ]:
# 1. Cài đặt các Package bổ sung (Pytorch framework)
%pip install segmentation-models-pytorch albumentations opencv-python matplotlib

In [ ]:
import os
import cv2
import torch
import numpy as np
import segmentation_models_pytorch as smp
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

# 2. Kiểm tra phần cứng
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔹 Đang sử dụng thiết bị: {device}")

In [ ]:
# 3. Khai báo Dataset cho Model U-Net
class CleaningMaskDataset(Dataset):
    def __init__(self, images_dir, masks_dir):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        
        # Quét tên file ảnh. Giả định ảnh .jpg và mask .png có cùng tên file
        self.ids = [f.split('.')[0] for f in os.listdir(images_dir) if f.endswith('.jpg') or f.endswith('.png')]
        
    def __getitem__(self, i):
        # Đọc ảnh gốc (RGB)
        img_name = self.ids[i]
        img_path = os.path.join(self.images_dir, f"{img_name}.jpg")
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (512, 512)) / 255.0  # Normalize
        image = np.transpose(image, (2, 0, 1))           # (C, H, W)
        image = torch.tensor(image, dtype=torch.float32)
        
        # Đọc ảnh Mask (Grayscale)
        # LƯU Ý: Nếu mask của bạn có đuôi khác (như .jpg), hãy chỉnh đuôi file bên dưới!
        mask_path = os.path.join(self.masks_dir, f"{img_name}.png") 
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (512, 512))
        mask = mask.astype('float32') / 255.0
        mask = np.expand_dims(mask, axis=0)              # (1, H, W)
        mask = torch.tensor(mask, dtype=torch.float32)
        
        return image, mask
        
    def __len__(self):
        return len(self.ids)

# ĐƯỜNG DẪN TỚI THƯ MỤC manual-dataset CỦA BẠN (Sửa để khớp ổ cứng)
TRAIN_IMGS = '../data/raw/unet/images' 
TRAIN_MSKS = '../data/raw/unet/masks'

if not os.path.exists(TRAIN_IMGS):
    print("⚠️ Chưa nạp dataset thủ công vào đường dẫn. Hãy trỏ tới đúng file ảnh của bạn!")
else:
    dataset = CleaningMaskDataset(TRAIN_IMGS, TRAIN_MSKS)
    print(f"✅ Khởi tạo DataLoader! Tổng số data tìm thấy: {len(dataset)} ảnh")
    train_loader = DataLoader(dataset, batch_size=8, shuffle=True) # batch=8 tối ưu cho RTX 3050

In [ ]:
# 4. Define Mô hình U-Net
try:
    model = smp.Unet(
        encoder_name="resnet18",        # Trọng số nền nhẹ, chạy bốc trên máy tính mỏng
        encoder_weights="imagenet",     # Pretrain trên ImageNet 
        in_channels=3,                  # Ảnh màu đầu vào RGB
        classes=1,                      # Nhị phân 0/1 (Sạch/Bẩn)
        activation='sigmoid'
    ).to(device)
    
    # Hàm Mất Mát & Opimizer
    loss_fn = torch.nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    print("✅ Khởi tạo U-Net thành công!")
except Exception as e:
    print("Chưa khởi chạy DataLoader ở block trên or chưa có data. Bỏ qua.")

In [ ]:
# 5. Chạy Vòng lặp Training thủ công (PyTorch Training Loop)
epochs = 20

# BẠN CÓ THỂ BỎ COMMENT Ở DƯỚI ĐỂ CHẠY KHI ĐÃ CÓ DATA NHÉ.

"""
model.train()
for epoch in range(epochs):
    total_loss = 0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        
        # Forward
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

    print(f'Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(train_loader):.4f}')

# Lưu weights
os.makedirs('../models', exist_ok=True)
torch.save(model.state_dict(), '../models/unet_poc.pth')
"""
print("Training loop đã sẵn sàng.")